In [ ]:
from xarray_utils import analyze_netcdf, zarr_to_netcdf, find_missing_days
import pandas as pd
import xarray as xr
import numpy as np


In [ ]:
# Open the Zarr dataset
ds_imd = xr.open_zarr("IMD_rainfall_0p25.zarr")
ds_imd = ds_imd.where(ds_imd != -999)

# Save as NetCDF
# ds_imd.to_netcdf("IMD_rainfall_0p25.nc")

In [ ]:
# Open the Zarr dataset
ds_ecm = xr.open_zarr("s2s_reforecast_sorted.zarr")

# Save as NetCDF
ds_ecm.to_netcdf("s2s_reforecast.nc")

In [ ]:
analyze_netcdf("IMD_rainfall_0p25.nc")


In [ ]:
analyze_netcdf("s2s_reforecast.nc")

In [ ]:
ds_ecm.data_vars

In [ ]:

# convert lead days into timedeltas
step_td = pd.to_timedelta(ds_ecm.step.values, unit="D").to_numpy()

ds_ecmv = ds_ecm.assign_coords(
    valid_time=(("time", "step"),
                ds_ecm.time.values[:, None] + step_td[None, :])
)


In [ ]:
print(ds_ecmv.coords)

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Lead": ds_ecmv.step.values,
    "Valid Date": ds_ecmv.valid_time.isel(time=2).dt.strftime("%Y-%m-%d").values
})

print(df)

In [ ]:
import numpy as np
import pandas as pd

target_date = np.datetime64("2005-01-03")

mask = ds_ecmv.valid_time == target_date

time_idx, step_idx = np.where(mask)

rows = []

for t, s in zip(time_idx, step_idx):
    rows.append({
        "Initialization": str(ds_ecmv.time.values[t])[:10],
        "Lead": int(ds_ecmv.step.values[s]),
        "Valid Date": str(ds_ecmv.valid_time.values[t, s])[:10]
    })

df = pd.DataFrame(rows)

print(df)

In [ ]:
missing_days = find_missing_days("s2s_reforecast.nc")

In [ ]:
ds_ecmv["total_precipitation"].isel(time=3719)

In [ ]:
print(ds_ecmv["total_precipitation"])

In [ ]:
for var in ds_ecmv.data_vars:
    print(f"{var}: {ds_ecmv[var].isnull().sum().values}")
    print(f"{var} min: {ds_ecmv[var].min().values}")
    print(f"{var} max: {ds_ecmv[var].max().values}")

In [ ]:
ds_imd.coords

In [ ]:
imd_sub = ds_imd.sel(time=slice('2005-01-01', '2025-02-11'))

In [ ]:
ds_ecmv.data_vars

In [ ]:
# import numpy as np

# samples = []

# for target_date in imd_sub.time.values:

#     mask = ds_ecmv.valid_time.astype("datetime64[D]") == target_date
#     time_idx, step_idx = np.where(mask)

#     # IMD target
#     y = imd_sub["rain"].sel(time=target_date).values

#     for t, s in zip(time_idx, step_idx):

#         sample = {
#             "date": target_date,
#             "lead": int(ds_ecmv.step.values[s]),
#             "target": y,
#             "10m_u_component_of_wind": ds_ecmv["10m_u_component_of_wind"].isel(time=t, step=s).values,
#             "10m_v_component_of_wind": ds_ecmv["10m_v_component_of_wind"].isel(time=t, step=s).values,
#             "2m_temperature": ds_ecmv["2m_temperature"].isel(time=t, step=s).values,
#             "mean_sea_level_pressure": ds_ecmv["mean_sea_level_pressure"].isel(time=t, step=s).values,
#             "specific_humidity_1000": ds_ecmv["specific_humidity_1000"].isel(time=t, step=s).values,
#             "specific_humidity_200": ds_ecmv["specific_humidity_200"].isel(time=t, step=s).values,
#             "specific_humidity_500": ds_ecmv["specific_humidity_500"].isel(time=t, step=s).values,
#             "specific_humidity_850": ds_ecmv["specific_humidity_850"].isel(time=t, step=s).values,
#             "temperature_1000": ds_ecmv["temperature_1000"].isel(time=t, step=s).values,
#             "temperature_200": ds_ecmv["temperature_200"].isel(time=t, step=s).values,
#             "temperature_500": ds_ecmv["temperature_500"].isel(time=t, step=s).values,
#             "temperature_850": ds_ecmv["temperature_850"].isel(time=t, step=s).values,
#             "total_precipitation": ds_ecmv["total_precipitation"].isel(time=t, step=s).values,
#             "u_component_of_wind_1000": ds_ecmv["u_component_of_wind_1000"].isel(time=t, step=s).values,
#             "u_component_of_wind_200": ds_ecmv["u_component_of_wind_200"].isel(time=t, step=s).values,
#             "u_component_of_wind_500": ds_ecmv["u_component_of_wind_500"].isel(time=t, step=s).values,
#             "u_component_of_wind_850": ds_ecmv["u_component_of_wind_850"].isel(time=t, step=s).values,
#             "v_component_of_wind_1000": ds_ecmv["v_component_of_wind_1000"].isel(time=t, step=s).values,
#             "v_component_of_wind_200": ds_ecmv["v_component_of_wind_200"].isel(time=t, step=s).values,
#             "v_component_of_wind_500": ds_ecmv["v_component_of_wind_500"].isel(time=t, step=s).values,
#             "v_component_of_wind_850": ds_ecmv["v_component_of_wind_850"].isel(time=t, step=s).values,
#         }

#         samples.append(sample)

In [ ]:
# pd.DataFrame(samples).to_csv("sampleout.csv", index=False)

## Build X / y for a series of lat-lons (pointwise downscaling)

For every requested `(lat, lon)`, every initialization `time`, and every `lead` (`step`),
we form one training row:

- **y** = IMD `rain` at that point on the *valid date* (`valid_time = time + lead`)
- **X** = the 21 S2S variables at that point + `lat`, `lon`, `lead`

The S2S grid is 1° (33×35) and IMD is 0.25° (129×135), so we select each point on
*its own* grid with nearest-neighbour, then **merge on `valid_time`** — this also handles
the case where several `(init, lead)` pairs forecast the same date.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

# ------------------------------------------------------------------
# 0. Open lazily (dask-backed). Without `chunks=`, xr.open_zarr loads
#    the ENTIRE ~15GB store into RAM as plain numpy on open, before
#    any .sel() happens — that alone was enough to crash the kernel.
# ------------------------------------------------------------------
ds_ecmv = xr.open_zarr("s2s_reforecast_sorted.zarr", chunks="auto")

_imd = xr.open_zarr("IMD_rainfall_0p25.zarr", chunks="auto")
_imd = _imd.where(_imd != -999)
imd_sub = _imd.sel(time=slice("2005-01-01", "2025-02-11"))

# ---- one-time diagnostics: confirm the IMD slice is non-empty ----
_it = imd_sub.time.values
print("imd_sub.time:", _it.dtype,
      str(_it.min())[:10] if _it.size else "EMPTY",
      "->", str(_it.max())[:10] if _it.size else "EMPTY", f"(n={_it.size})")

# ------------------------------------------------------------------
# 1. Build the point list from IMD LAND cells only, then subsample.
#    IMD rainfall is land-only: ~72% of the 129x135 grid is ocean/NaN.
#    A cell is "land" if it has >=1 valid rain obs. For the baseline we
#    take every 4th land cell (SUBSAMPLE) to keep the run tractable
#    while preserving national coverage.
# ------------------------------------------------------------------
land_mask = imd_sub["rain"].notnull().any("time").compute()   # (lat, lon) bool
lat_idx, lon_idx = np.where(land_mask.values)
points = [
    (float(imd_sub.lat.values[i]), float(imd_sub.lon.values[j]))
    for i, j in zip(lat_idx, lon_idx)
]

SUBSAMPLE = 4                 # 1/4 of land cells; set to 1 for the full grid
points = points[::SUBSAMPLE]
print(f"land points after 1/{SUBSAMPLE} subsample: {len(points)}")

PREDICTORS = list(ds_ecmv.data_vars)   # all 21 S2S variables
feature_cols = ["lat", "lon", "lead"] + PREDICTORS

# ------------------------------------------------------------------
# 2. Process points in small batches, streaming to CSV.
# ------------------------------------------------------------------
POINTS_PER_BATCH = 25   # lower this further if memory is still tight
OUT_PATH = Path("samples.csv")
OUT_PATH.unlink(missing_ok=True)

def build_batch(batch_points, debug=False):
    lat_pts = xr.DataArray([p[0] for p in batch_points], dims="point")
    lon_pts = xr.DataArray([p[1] for p in batch_points], dims="point")

    # predictors at each point -> (time, step, point); drop the nearest-grid
    # lat/lon coords so they don't collide with the requested lat/lon later.
    ecm_pts = (
        ds_ecmv[PREDICTORS]
        .sel(lat=lat_pts, lon=lon_pts, method="nearest")
        .drop_vars(["lat", "lon"])
    )
    bdf = ecm_pts.to_dataframe().reset_index().rename(columns={"step": "lead"})
    # reconstruct valid_time IN PANDAS (robust: does not depend on a 2D xarray
    # coordinate surviving to_dataframe under dask).
    bdf["valid_time"] = bdf["time"] + pd.to_timedelta(bdf["lead"], unit="D")

    imd_pts = (
        imd_sub["rain"]
        .sel(lat=lat_pts, lon=lon_pts, method="nearest")
        .drop_vars(["lat", "lon"])
    )
    tdf = (
        imd_pts.to_dataframe()
        .reset_index()
        .rename(columns={"time": "valid_time", "rain": "target"})
    )

    merged = bdf.merge(tdf, on=["valid_time", "point"], how="left")

    if debug:
        print("  [debug] ecm rows:", len(bdf), " imd rows:", len(tdf),
              " imd target non-null:", int(tdf["target"].notna().sum()))
        print("  [debug] merged target non-null:", int(merged["target"].notna().sum()), "/", len(merged))

    pt_coords = pd.DataFrame({
        "point": range(len(batch_points)),
        "lat": [p[0] for p in batch_points],
        "lon": [p[1] for p in batch_points],
    })
    merged = merged.merge(pt_coords, on="point").drop(columns="point")
    merged = merged.dropna(subset=["target"]).reset_index(drop=True)
    return merged[feature_cols + ["valid_time", "target"]]

n_batches = int(np.ceil(len(points) / POINTS_PER_BATCH))
total_rows = 0
for i in range(n_batches):
    batch_points = points[i * POINTS_PER_BATCH : (i + 1) * POINTS_PER_BATCH]
    bdf = build_batch(batch_points, debug=(i == 0))
    bdf.to_csv(OUT_PATH, mode="a", header=(i == 0), index=False)
    total_rows += len(bdf)
    if i % 20 == 0 or i == n_batches - 1:
        print(f"batch {i + 1}/{n_batches}: +{len(bdf)} rows (total {total_rows})")

assert total_rows > 0, (
    "0 rows: merge matched nothing. If '[debug] imd target non-null' is 0 the "
    "points are all ocean/NaN cells."
)

# ------------------------------------------------------------------
# 3. Done. Samples are on disk in OUT_PATH. We do NOT load the whole
#    file here — it can be tens of GB. The training cell reads it in
#    chunks. Preview the first rows only.
# ------------------------------------------------------------------
print(f"\nwrote {total_rows} rows to {OUT_PATH} ({OUT_PATH.stat().st_size / 1e9:.2f} GB)")
pd.read_csv(OUT_PATH, nrows=5, parse_dates=["valid_time"])

imd_sub.time: datetime64[ns] 2005-01-01 -> 2025-02-11 (n=7347)
land points after 1/4 subsample: 1241
  [debug] ecm rows: 3999000  imd rows: 183675  imd target non-null: 183675
  [debug] merged target non-null: 3999000 / 3999000
batch 1/50: +3999000 rows (total 3999000)


In [ ]:
df.to_csv("s2s_samples.csv", index=False)

## Baseline: Linear Regression for lead days 14–42

Subweek-3/4-to-subseasonal (S2S) range. We:

1. Filter `df` to `14 <= lead <= 42`.
2. Split **chronologically** by `valid_time` (not randomly) — a random split would put
   nearby dates/leads in both train and test, leaking autocorrelated rain into "unseen" data.
3. Standardize features (pressure ~1e5, humidity ~1e-3 — raw linear regression would be
   dominated by scale) and fit `LinearRegression`.
4. Compare against a `DummyRegressor(strategy="mean")` computed on the *same* train/test
   split, so the comparison is apples-to-apples with the 14–42 day subset (not the whole
   dataset's mean).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# Out-of-core linear regression for lead days 14-42.
# The samples CSV can be tens of GB, so we NEVER load it whole. We
# stream it in chunks and accumulate the standardized OLS normal
# equations (X^T X, X^T y) -> this gives the EXACT least-squares fit,
# is single-machine, memory-bounded, and order-independent.
# ------------------------------------------------------------------
OUT_PATH = Path("samples.csv")
LEAD_MIN, LEAD_MAX = 14, 42
CUTOFF = pd.Timestamp("2021-01-01")   # chronological train/test split (~80/20)
CHUNK = 1_000_000                     # rows per read; lower if memory is tight

# feature columns from the CSV header (independent of the builder cell's state)
header = pd.read_csv(OUT_PATH, nrows=0).columns.tolist()
feature_cols = [c for c in header if c not in ("valid_time", "target")]
p = len(feature_cols)

def stream(before):
    """Yield lead-14-42 rows on the train (valid_time < CUTOFF) or test side."""
    for ch in pd.read_csv(OUT_PATH, chunksize=CHUNK, parse_dates=["valid_time"]):
        lead_ok = ch["lead"].between(LEAD_MIN, LEAD_MAX)
        m = lead_ok & (ch["valid_time"] < CUTOFF if before else ch["valid_time"] >= CUTOFF)
        if m.any():
            yield ch.loc[m]

# ---- Pass 1: train feature mean / std (for standardization) ----
n = 0; s = np.zeros(p); s2 = np.zeros(p)
for ch in stream(before=True):
    Xc = ch[feature_cols].to_numpy(np.float64)
    n += len(Xc); s += Xc.sum(0); s2 += (Xc * Xc).sum(0)
mean = s / n
std = np.sqrt(np.maximum(s2 / n - mean**2, 1e-12))
print(f"train rows (lead {LEAD_MIN}-{LEAD_MAX}, < {CUTOFF.date()}): {n}")

# ---- Pass 2: accumulate standardized normal equations -> exact OLS ----
XtX = np.zeros((p + 1, p + 1)); Xty = np.zeros(p + 1); sum_y = 0.0
for ch in stream(before=True):
    Xc = ch[feature_cols].to_numpy(np.float64)
    yc = ch["target"].to_numpy(np.float64)
    Z1 = np.hstack([np.ones((len(Xc), 1)), (Xc - mean) / std])   # intercept + standardized
    XtX += Z1.T @ Z1
    Xty += Z1.T @ yc
    sum_y += yc.sum()
beta = np.linalg.lstsq(XtX, Xty, rcond=None)[0]   # [intercept, standardized slopes...]
ybar_train = sum_y / n                             # climatology baseline

# ---- Pass 3: streaming test metrics for model vs climatology ----
n_t = 0; sum_yt = 0.0; sum_yt2 = 0.0
sse = 0.0; sae = 0.0; sse_base = 0.0
for ch in stream(before=False):
    Xc = ch[feature_cols].to_numpy(np.float64)
    yc = ch["target"].to_numpy(np.float64)
    pred = np.hstack([np.ones((len(Xc), 1)), (Xc - mean) / std]) @ beta
    r = yc - pred
    sse += (r * r).sum(); sae += np.abs(r).sum()
    sse_base += ((yc - ybar_train) ** 2).sum()
    n_t += len(yc); sum_yt += yc.sum(); sum_yt2 += (yc * yc).sum()

var_t = sum_yt2 / n_t - (sum_yt / n_t) ** 2   # test-target variance
r2 = 1 - sse / (var_t * n_t)
rmse = np.sqrt(sse / n_t); mae = sae / n_t
r2_base = 1 - sse_base / (var_t * n_t)
rmse_base = np.sqrt(sse_base / n_t)

print(f"test rows: {n_t}  (split at {CUTOFF.date()})")
print(f"\nLinear Regression  -> R2: {r2:.4f}  RMSE: {rmse:.4f}  MAE: {mae:.4f}")
print(f"Climatology (mean) -> R2: {r2_base:.4f}  RMSE: {rmse_base:.4f}")

# coefficients on standardized features (comparable across variables)
coef = pd.Series(beta[1:], index=feature_cols).sort_values(key=np.abs, ascending=False)
print("\nTop standardized coefficients:")
print(coef.head(10).to_string())

In [ ]:
samples[20]

In [ ]:
samples.dtype

In [ ]:
from sklearn.dummy import DummyRegressor

if len(X) < 2:
    print("Not enough valid samples to train a baseline model.")
else:
    baseline = DummyRegressor(strategy="mean")
    baseline.fit(X_train, y_train)

    baseline_pred = baseline.predict(X_test)
    baseline_r2 = r2_score(y_test, baseline_pred)
    baseline_rmse = root_mean_squared_error(y_test, baseline_pred)

    print(f"Baseline Test R^2: {baseline_r2:.4f}")
    print(f"Baseline Test RMSE: {baseline_rmse:.4f}")


In [ ]:
rain_mean = float(ds_imd["rain"].mean(skipna=True).values)
print(f"Mean rainfall: {rain_mean:.4f}")


In [ ]:
y[4]